In [ ]:
import sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

sys.path.append(str(Path.cwd() / "src"))
from boosting import GradientBoostingScratch

# ---------- Data-generating process ----------
# 8 features, uniform on [-1, 1]. The signal mixes a LINEAR part (features 0-3) and a
# NONLINEAR part (features 4-7: an interaction, a symmetric band, and an even cosine).
# Each part is standardised, so the signal variance is the same for every mixing weight a:
#     y = sqrt(1 - a) * linear + sqrt(a) * nonlinear + noise
# The nonlinear terms are even or product functions, so they have zero linear correlation
# with every feature: a linear model cannot capture them at all.
BETA = np.array([1.0, -1.0, 0.5, -0.5])
SIGMA = 0.5


def parts(X):
    lin = X[:, :4] @ BETA
    non = X[:, 4] * X[:, 5] + (np.abs(X[:, 6]) > 0.5) + np.cos(2 * np.pi * X[:, 7])
    return lin, non


ref = np.random.default_rng(12345).uniform(-1, 1, size=(200_000, 8))
l_ref, n_ref = parts(ref)
L_MEAN, L_STD, N_MEAN, N_STD = l_ref.mean(), l_ref.std(), n_ref.mean(), n_ref.std()


def make_data(n, a, rng):
    X = rng.uniform(-1, 1, size=(n, 8))
    lin, non = parts(X)
    lin, non = (lin - L_MEAN) / L_STD, (non - N_MEAN) / N_STD
    return X, np.sqrt(1 - a) * lin + np.sqrt(a) * non + SIGMA * rng.normal(size=n)


rmse = lambda a, b: float(np.sqrt(mean_squared_error(a, b)))


def boost_early_stop(X_tr, y_tr, seed, max_trees=200, lr=0.1, depth=3):
    X_a, X_v, y_a, y_v = train_test_split(X_tr, y_tr, test_size=0.2, random_state=seed)
    m = GradientBoostingScratch("regression", n_estimators=max_trees, learning_rate=lr, max_depth=depth).fit(X_a, y_a)
    best = int(np.argmin([mean_squared_error(y_v, F) for F in m.staged_decision_function(X_v)])) + 1
    return GradientBoostingScratch("regression", n_estimators=best, learning_rate=lr, max_depth=depth).fit(X_tr, y_tr), best


# ---------- Experiment ----------
N_TRAIN, N_TEST, REPS = 400, 3000, 10
A_VALUES = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
rows, t0 = [], time.time()
for a in A_VALUES:
    for rep in range(REPS):
        rng = np.random.default_rng(1000 * rep + int(round(a * 10)))      # independent dataset per (a, rep)
        X_tr, y_tr = make_data(N_TRAIN, a, rng)
        X_te, y_te = make_data(N_TEST, a, rng)
        lin = LinearRegression().fit(X_tr, y_tr)
        gb, best = boost_early_stop(X_tr, y_tr, seed=rep)
        rows.append({"a": a, "rep": rep, "trees": best,
                     "RMSE linear": rmse(y_te, lin.predict(X_te)),
                     "RMSE boosting": rmse(y_te, gb.predict(X_te)),
                     "RMSE noise floor": SIGMA})
    print(f"a = {a:.1f} done ({time.time() - t0:.0f}s)")
res = pd.DataFrame(rows)
res["diff (boosting - linear)"] = res["RMSE boosting"] - res["RMSE linear"]


# ---------- Summary with ordinary t-intervals (the repetitions are independent datasets) ----------
def ci(x):
    x = np.asarray(x)
    half = stats.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x))
    return x.mean(), x.mean() - half, x.mean() + half

out = []
for a, g in res.groupby("a"):
    m, lo, hi = ci(g["diff (boosting - linear)"])
    out.append({"a (nonlinear share)": a,
                "RMSE linear": g["RMSE linear"].mean(), "RMSE boosting": g["RMSE boosting"].mean(),
                "diff (boosting - linear)": m, "95% CI low": lo, "95% CI high": hi,
                "boosting better?": "yes" if hi < 0 else ("linear better" if lo > 0 else "unclear"),
                "median trees": int(g["trees"].median())})
summary = pd.DataFrame(out)
print(f"\nNoise floor (best possible RMSE) = {SIGMA}.  n_train = {N_TRAIN}, {REPS} independent datasets per setting.")
display(summary.round(3).set_index("a (nonlinear share)"))

# ---------- Figure ----------
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
g = res.groupby("a")
for col, lab in [("RMSE linear", "Linear regression"), ("RMSE boosting", "Gradient boosting (mine)")]:
    axes[0].errorbar(summary["a (nonlinear share)"], g[col].mean(), yerr=g[col].std(ddof=1) / np.sqrt(REPS) * 2,
                     marker="o", capsize=3, label=lab)
axes[0].axhline(SIGMA, color="gray", ls="--", label="noise floor")
axes[0].set_xlabel("Share of the signal that is nonlinear (a)")
axes[0].set_ylabel("Test RMSE"); axes[0].set_title("Test error vs nonlinearity"); axes[0].legend()

axes[1].errorbar(summary["a (nonlinear share)"], summary["diff (boosting - linear)"],
                 yerr=[summary["diff (boosting - linear)"] - summary["95% CI low"],
                       summary["95% CI high"] - summary["diff (boosting - linear)"]],
                 marker="o", capsize=3, color="tab:green")
axes[1].axhline(0, color="black", lw=1)
axes[1].set_xlabel("Share of the signal that is nonlinear (a)")
axes[1].set_ylabel("RMSE(boosting) - RMSE(linear)")
axes[1].set_title("Below zero = boosting wins (95% CI)")
plt.tight_layout()
Path("reports/figures").mkdir(parents=True, exist_ok=True)
plt.savefig("reports/figures/fig4_crossover.png", dpi=200)
plt.show()
res.to_csv("reports/synthetic_crossover_results.csv", index=False)
summary.to_csv("reports/synthetic_crossover_summary.csv", index=False)
print("Saved reports/figures/fig4_crossover.png and two CSV files in reports/")